# 🚀 Supervised Classification: Inference Latency & Auto-Features

### 📦 Installation
Ensure the package is installed in your environment:
```bash
pip install -e .
```

### 🎯 Overview
In this notebook, we explore the `FastKMeansClassifier` on the massive real-world **Forest Covertype** dataset.
Standard algorithms like `KNeighborsClassifier` are "lazy" learners. They store the entire dataset and compute distances to *every single point* during inference, resulting in an $\mathcal{O}(N_{train} \times D)$ complexity. This makes them unusably slow in production.

`FastKMeansClassifier` compresses the manifold into $K$ intelligent prototypes, dropping inference complexity to **$\mathcal{O}(K \times D)$**. 

Furthermore, we showcase:
1. **Auto Feature Weights (ANOVA):** Automatically calculates within-cluster variance to weight features algorithmically.
2. **Negative Sampling:** Speeds up the gradient phase by skipping loss calculations for unrepresented negative classes.

In [13]:
import time
import pandas as pd
import torch
import numpy as np
from sklearn.datasets import fetch_covtype
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from IPython.display import display

from fast_kmeans import FastKMeansClassifier

# 1. Load Massive Dataset (Forest Covertypes - Sampled to 50k for interactive testing)
print("Fetching Forest Covertype dataset...")
data = fetch_covtype()
idx = np.random.choice(len(data.data), 50000, replace=False)
# Targets are 1-7, we shift to 0-6 for standard zero-indexed PyTorch mapping
X_np, y_np = data.data[idx], data.target[idx] - 1  

X_scaled = StandardScaler().fit_transform(X_np)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_np, test_size=0.2, random_state=42)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)

print("🚀 Benchmarking FastKMeansClassifier vs Standard KNN...\n")

# --- SKLEARN K-NEAREST NEIGHBORS ---
# KNN trains instantly (just saves data in memory), but inference is highly inefficient.
knn = KNeighborsClassifier(n_neighbors=5, metric='euclidean', n_jobs=-1)
knn.fit(X_train, y_train)

start = time.time()
knn_preds = knn.predict(X_test)
knn_inf_time = time.time() - start
knn_acc = accuracy_score(y_test, knn_preds)

# --- FAST-KMEANS CLASSIFIER ---
fkm = FastKMeansClassifier(
    k_init='auto', # 100 Prototypes per class
    distance='cosine', 
    auto_feature_weights=True, # Dynamic ANOVA weighting
    diversity_reg=0.01,
    repulsion_factor=0.01
)
# Training takes time to build the compressed topology
fkm.fit(X_train_t, y_train_t, max_iters=20, verbose=True)
fkm.finetune(X_train_t, y_train_t, epochs=30, eval_fraction=0.2, lr=0.1, verbose=True)

start = time.time()
fkm_preds = fkm.predict(X_test_t).cpu().numpy()
fkm_inf_time = time.time() - start
fkm_acc = accuracy_score(y_test, fkm_preds)

# 2. Build Comparison Table
results = pd.DataFrame({
    "Algorithm": ["Sklearn KNN (k=5)", "FastKMeansClassifier (k_init=100)"],
    "Stored Parameters": ["Full Dataset (40,000 vectors)", "Compressed Grid (700 Prototypes)"],
    "Inference Complexity": ["O(N_train * D)", "O(K * D)"],
    "Accuracy": [f"{knn_acc:.4f}", f"{fkm_acc:.4f}"],
    "Inference Time (s)": [f"{knn_inf_time:.4f}", f"{fkm_inf_time:.4f}"]
})

display(results.style.set_caption("Supervised Classification: Inference Latency & Accuracy")
        .set_properties(**{'text-align': 'center'}))

Fetching Forest Covertype dataset...
🚀 Benchmarking FastKMeansClassifier vs Standard KNN...



Fit (EMA Topology):   0%|          | 0/20 [00:00<?, ?it/s]

Finetune (Gradient):   0%|          | 0/30 [00:00<?, ?it/s]

,Algorithm,Stored Parameters,Inference Complexity,Accuracy,Inference Time (s)
0,Sklearn KNN (k=5),"Full Dataset (40,000 vectors)",O(N_train * D),0.8255,0.5981
1,FastKMeansClassifier (k_init=100),Compressed Grid (700 Prototypes),O(K * D),0.6859,0.0483
